
Para **cada target separadamente**:

### Variáveis de decisão

$$z_k \in {0,1}, \quad k = 1,\dots,10 $$

* (z_k = 1) → o preditor (k) é usado
* (z_k = 0) → o preditor é descartado

### Restrição opcional (controle de complexidade)

$$\sum_{k=1}^{10} z_k \leq K_{\max} $$

Ex.: no máximo 4 ou 5 entradas.

### Função objetivo

$$\max \quad R^2_{\text{test}}(\mathbf{z}) $$



* **Medições físicas/químicas (entradas)**
  $$  \mathbf{x}_i = [\text{pH}, \text{Cond}, \text{Temp}, \dots, \text{Cor}] \in \mathbb{R}^{10}
$$

* **Concentrações químicas (saídas)**
  $$  \mathbf{y}_i = [\text{Fe}, \text{Al}, \dots, \text{Mn}] \in \mathbb{R}^{10}
$$

Cada amostra é um ponto:
$$(\mathbf{x}_i, \mathbf{y}_i)$$

---

## 🔹 Agora entra a decisão (variáveis binárias)

Você introduz um **vetor de decisão**:

$$\mathbf{z} = [z_1, z_2, \dots, z_{10}], \quad z_k \in {0,1}$$

onde:

* (z_k = 1) → o preditor (k) é usado
* (z_k = 0) → o preditor (k) é descartado

👉 Esse vetor **não vem dos dados**, ele é decidido pelo algoritmo genético.

---

## 🔹 Como os dados mudam com `z`

Dado um vetor `z`, você **projeta os dados**:

$$\mathbf{x}*i^{(z)} = { x*{ik} ;|; z_k = 1 }$$

Exemplo:

```
z = [1, 0, 0, 1, 0, 0, 0, 1, 0, 0]
```

Significa:

```
Entradas usadas = [pH, OD, ORP]
```

Então:

```python
X_sel = X[:, mask == 1]
```

🔴 Importante:

* **Nenhum valor é modificado**
* Você **apenas remove colunas**


In [2]:
import pandas as pd
import numpy as np 
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.gaussian_process import GaussianProcessRegressor as gpr
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from deap import base, creator, tools, algorithms
import random
import os

In [14]:
PREDICTORS = ["pH", "Cond", "Temp", "OD", "Tds", "Resist", "Salin", "ORP", "IP", "Cor"] # 10 entradas
TARGETS = ["Fe", "Al", "As", "Pb", "Zn", "Hg", "Co", "V", "Ba", "Mn"] # 10 saidas

SCALER = StandardScaler()
OUT_SCALER = StandardScaler()

In [4]:
Dataset = pd.read_excel("../Dados/Dados.xlsx")

Datasets = []

for n in range(1, 6):
    n_data = Dataset[ Dataset["Pontos"] == f"P{n}" ]
    n_data = n_data.drop(columns=["Campanhas", "Pontos"])
    
    n_data_norm = n_data
        
    Datasets.append(n_data)

In [5]:
def ComputeMetrics(y_train, y_train_pred, y_test, y_test_pred):
    return {
        "mse_train": mean_squared_error(y_train, y_train_pred),
        "r2_train":  r2_score(y_train, y_train_pred),
        "mse_test":  mean_squared_error(y_test, y_test_pred),
        "r2_test":   r2_score(y_test, y_test_pred)
    }

In [6]:
GPR_PARAMS = {
    "Fe": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-8},
    "Al": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "As": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Pb": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "Zn": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e9, "alpha": 1e-3},
    "Hg": {"nu": 0.5, "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Co": {"nu": 0.5,  "ls_min": 1e-6, "ls_max": 1e10, "alpha": 1e-3},
    "V":  {"nu": 0.25, "ls_min": 1e-6, "ls_max": 1e6, "alpha": 1e-3},
    "Ba": {"nu": 0.5,  "ls_min": 1e-8, "ls_max": 1e8, "alpha": 1e-3},
    "Mn": {"nu": 1.5,  "ls_min": 1e-8, "ls_max": 1e8, "alpha": 1e-3},
}

In [7]:

def GprModel(X_train_pca, X_test_pca, y_train, y_test, target, n):

    params = GPR_PARAMS[target]

    kernel = C(1.0, (1e-3, 1e3)) + C(1.0) * Matern(
        length_scale=np.ones(X_train_pca.shape[1]),
        nu=params["nu"],
        length_scale_bounds=(params["ls_min"], params["ls_max"])
    )

    model = gpr(
        kernel=kernel,
        alpha=params["alpha"],
        normalize_y=False,
        n_restarts_optimizer=20
    )

   # Treinamento
    model.fit(X_train_pca, y_train)

    # Predições
    y_train_pred = model.predict(X_train_pca)
    y_test_pred = model.predict(X_test_pca)

    # Desnormalização
    y_train_denorm = OUT_SCALER.inverse_transform(y_train.reshape(-1, 1)).ravel()
    y_train_pred_denorm = OUT_SCALER.inverse_transform(y_train_pred.reshape(-1, 1)).ravel()

    y_test_denorm = OUT_SCALER.inverse_transform(y_test.reshape(-1, 1)).ravel()
    y_test_pred_denorm = OUT_SCALER.inverse_transform(y_test_pred.reshape(-1, 1)).ravel()
    
    metrics = ComputeMetrics(y_train_denorm, y_train_pred_denorm,  y_test_denorm, y_test_pred_denorm)
    
    return  model, metrics

In [8]:
def fitness_function(mask, X, Y, target_idx):
    """
    mask: array binário (0/1) com tamanho = n_predictors
    """

    # Penaliza solução vazia
    if mask.sum() == 0:
        return -1.0

    X_sel = X[:, mask == 1]

    X_train, X_test, Y_train, Y_test = train_test_split(
        X_sel, Y[:, target_idx], test_size=0.2, random_state=42
    )

    x_train = SCALER.fit_transform(X_train)
    x_test  = SCALER.transform(X_test)

    y_train = OUT_SCALER.fit_transform(Y_train.reshape(-1, 1)).ravel()
    y_test  = OUT_SCALER.transform(Y_test.reshape(-1, 1)).ravel()

    _, metrics = GprModel(
        x_train, x_test,
        y_train, y_test,
        TARGETS[target_idx],
        n=0
    )

    return metrics["r2_test"]


In [9]:
if not hasattr(creator, "FitnessMax"):
    creator.create("FitnessMax", base.Fitness, weights=(1.0,))
if not hasattr(creator, "Individual"):
    creator.create("Individual", list, fitness=creator.FitnessMax)


In [20]:
def run_ga_gpr_for_target(
    Datasets,
    PREDICTORS,
    TARGETS,
    target,
    ponto,
    excel_file="GA-GPR.xlsx",
    min_predictors=2,
    max_predictors=3,
    random_state=42
):

    # Dataset correspondente ao ponto
    Dataset = Datasets[ponto - 1]

    X = Dataset[PREDICTORS].values
    Y = Dataset[TARGETS].values

    j = TARGETS.index(target)

    print(f"\n=== P{ponto} | Target: {target} ===")

    toolbox = base.Toolbox()
    toolbox.register("attr_bool", random.randint, 0, 1)
    toolbox.register(
        "individual",
        tools.initRepeat,
        creator.Individual,
        toolbox.attr_bool,
        n=len(PREDICTORS)
    )
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)

    # Função de avaliação do GA
    def evaluate(individual):
        mask = np.array(individual)

        # Restrição: mínimo e máximo de preditores
        if mask.sum() < min_predictors or mask.sum() > max_predictors:
            return (-1.0,)

        r2 = fitness_function(mask, X, Y, j)
        return (r2,)

    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", tools.cxTwoPoint)
    toolbox.register("mutate", tools.mutFlipBit, indpb=0.1)
    toolbox.register("select", tools.selTournament, tournsize=3)

    population = toolbox.population(n=40)

    algorithms.eaSimple(
        population,
        toolbox,
        cxpb=0.7,
        mutpb=0.2,
        ngen=30,
        verbose=False
    )

    # Melhor indivíduo
    best_ind = tools.selBest(population, 1)[0]
    best_mask = np.array(best_ind)
    best_predictors = [p for p, m in zip(PREDICTORS, best_mask) if m == 1]

    print("Entradas:", best_predictors)
    print("R2_test (GA):", best_ind.fitness.values[0])

    # Seleção das variáveis
    X_sel = X[:, best_mask == 1]

    X_train, X_test, y_train, y_test = train_test_split(
        X_sel, Y[:, j], test_size=0.2, random_state=random_state
    )

    x_train = SCALER.fit_transform(X_train)
    x_test  = SCALER.transform(X_test)

    y_train_s = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
    y_test_s  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()

    _, metrics = GprModel(
        x_train, x_test,
        y_train_s, y_test_s,
        target,
        n=ponto
    )

    # Linha de resultados
    row = {
        "P": ponto,
        "target": target,
        "n_predictors": int(best_mask.sum()),
        "predictors": "; ".join(best_predictors),
        "r2_train": metrics["r2_train"],
        "mse_train": metrics["mse_train"],
        "r2_test": metrics["r2_test"],
        "mse_test": metrics["mse_test"]
    }

    df_new = pd.DataFrame([row])

    if os.path.exists(excel_file):
        df_old = pd.read_excel(excel_file)
        df_final = pd.concat([df_old, df_new], ignore_index=True)
    else:
        df_final = df_new

    df_final.to_excel(excel_file, index=False)

    return row


In [13]:
run_ga_gpr_for_target(
    Datasets=Datasets,
    PREDICTORS=PREDICTORS,
    TARGETS=TARGETS,
    target="Ba",
    ponto=3
)



=== P3 | Target: Ba ===


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-

Entradas: ['Temp', 'Resist', 'Salin', 'Cor']
R2_test (GA): 0.14493010129696648


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__length_scale is close to the specified upper bound 100000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(


{'P': 3,
 'target': 'Ba',
 'n_predictors': 4,
 'predictors': 'Temp; Resist; Salin; Cor',
 'r2_train': 0.9999986459991603,
 'mse_train': 0.00022861275553336252,
 'r2_test': 0.1449041247521594,
 'mse_test': 44.61005163831075}

In [18]:
run_ga_gpr_for_target(
    Datasets=Datasets,
    PREDICTORS=PREDICTORS,
    TARGETS=TARGETS,
    target="Ba",
    ponto=3
)



=== P3 | Target: Ba ===


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-

Entradas: ['Temp', 'Resist', 'Salin', 'Cor']
R2_test (GA): 0.14494448869372412


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


{'P': 3,
 'target': 'Ba',
 'n_predictors': 4,
 'predictors': 'Temp; Resist; Salin; Cor',
 'r2_train': 0.9999986459364428,
 'mse_train': 0.0002286233449048131,
 'r2_test': 0.1449339657916512,
 'mse_test': 44.60849484175588}

In [16]:
run_ga_gpr_for_target(
    Datasets=Datasets,
    PREDICTORS=PREDICTORS,
    TARGETS=TARGETS,
    target="As",
    ponto=5
)


=== P5 | Target: As ===


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 1 of parameter k2__k2__length_scale is close to the specified upper bound 1000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__k2__length_scale is close to the specified upper bound 1000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-e

Entradas: ['IP', 'Cor']
R2_test (GA): 0.33565387492988086


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


{'P': 5,
 'target': 'As',
 'n_predictors': 2,
 'predictors': 'IP; Cor',
 'r2_train': 0.9999985314664581,
 'mse_train': 3.741925996655136e-08,
 'r2_test': 0.3356538728627412,
 'mse_test': 0.013809599096922013}

In [21]:
run_ga_gpr_for_target(
    Datasets=Datasets,
    PREDICTORS=PREDICTORS,
    TARGETS=TARGETS,
    target="Ba",
    ponto=3
)



=== P3 | Target: Ba ===


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-

Entradas: ['Cond', 'Salin', 'Cor']
R2_test (GA): 0.1448962583003599


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


{'P': 3,
 'target': 'Ba',
 'n_predictors': 3,
 'predictors': 'Cond; Salin; Cor',
 'r2_train': 0.9999986460467747,
 'mse_train': 0.00022860471620035676,
 'r2_test': 0.14489956159846884,
 'mse_test': 44.610289696436936}

In [22]:
run_ga_gpr_for_target(
    Datasets=Datasets,
    PREDICTORS=PREDICTORS,
    TARGETS=TARGETS,
    target="As",
    ponto=5
)


=== P5 | Target: As ===


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 2 of parameter k2__k2__length_scale is close to the specified upper bound 1000000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib

Entradas: ['IP', 'Cor']
R2_test (GA): 0.3356538728627412


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


{'P': 5,
 'target': 'As',
 'n_predictors': 2,
 'predictors': 'IP; Cor',
 'r2_train': 0.9999985314664581,
 'mse_train': 3.741925996655136e-08,
 'r2_test': 0.3356538728627412,
 'mse_test': 0.013809599096922013}

In [ ]:
# excel_file = "GA-GPR.xlsx"

# for i, Dataset in enumerate(Datasets[2:]):

#     X = Dataset[PREDICTORS].values
#     Y = Dataset[TARGETS].values

#     for j, target in enumerate(TARGETS):

#         print(f"\n=== P{i+3} | Target: {target} ===")

#         toolbox = base.Toolbox()
#         toolbox.register("attr_bool", random.randint, 0, 1)
#         toolbox.register(
#             "individual",
#             tools.initRepeat,
#             creator.Individual,
#             toolbox.attr_bool,
#             n=len(PREDICTORS)
#         )
#         toolbox.register("population", tools.initRepeat, list, toolbox.individual)

#         def evaluate(individual):
#             mask = np.array(individual)

#             if mask.sum() == 0 or mask.sum() > 5:
#                 return (-1.0,)

#             r2 = fitness_function(mask, X, Y, j)
#             return (r2,)

#         toolbox.register("evaluate", evaluate)
#         toolbox.register("mate", tools.cxTwoPoint)
#         toolbox.register("mutate", tools.mutFlipBit, indpb=0.1)
#         toolbox.register("select", tools.selTournament, tournsize=3)

#         population = toolbox.population(n=40)

#         algorithms.eaSimple(
#             population,
#             toolbox,
#             cxpb=0.7,
#             mutpb=0.2,
#             ngen=30,
#             verbose=False
#         )

#         best_ind = tools.selBest(population, 1)[0]
#         best_mask = np.array(best_ind)
#         best_predictors = [p for p, m in zip(PREDICTORS, best_mask) if m == 1]

#         print("Entradas:", best_predictors)
#         print("R2_test (GA):", best_ind.fitness.values[0])
        
#         X_sel = X[:, best_mask == 1]

#         X_train, X_test, y_train, y_test = train_test_split(
#             X_sel, Y[:, j], test_size=0.2, random_state=42
#         )

#         x_train = SCALER.fit_transform(X_train)
#         x_test  = SCALER.transform(X_test)

#         y_train_s = OUT_SCALER.fit_transform(y_train.reshape(-1, 1)).ravel()
#         y_test_s  = OUT_SCALER.transform(y_test.reshape(-1, 1)).ravel()

#         _, metrics = GprModel(
#             x_train, x_test,
#             y_train_s, y_test_s,
#             target,
#             n=i+3
#         )

#         row = {
#             "P": i + 3,
#             "target": target,
#             "n_predictors": int(best_mask.sum()),
#             "predictors": "; ".join(best_predictors),
#             "r2_train": metrics["r2_train"],
#             "mse_train": metrics["mse_train"],
#             "r2_test": metrics["r2_test"],
#             "mse_test": metrics["mse_test"]
#         }

#         df_new = pd.DataFrame([row])

#         if os.path.exists(excel_file):
#             df_old = pd.read_excel(excel_file)
#             df_final = pd.concat([df_old, df_new], ignore_index=True)
#         else:
#             df_final = df_new

#         df_final.to_excel(excel_file, index=False)
#     break


=== P3 | Target: V ===


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\_gpr.py:663: ConvergenceWarning: lbfgs failed to converge after 16 iteration(s) (status=2):
ABNORMAL: 

You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
  _check_optimize_result("lbfgs", opt_res)
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-

Entradas: ['Cond', 'Temp', 'ORP', 'IP', 'Cor']
R2_test (GA): 0.603735338044836


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



=== P3 | Target: Ba ===


...

d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:450: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k1__constant_value is close to the specified upper bound 100000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__k2__length_scale is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



=== P3 | Target: Mn ===


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-

Entradas: ['Cond', 'Temp', 'OD', 'Tds']
R2_test (GA): 0.7005941871907557


d:\Users\sarah\Educ\Ic-2025.2\tf-env\lib\site-packages\sklearn\gaussian_process\kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__constant_value is close to the specified lower bound 0.001. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(
